# YgoMaster-PackBuilder — Domain Layer Tests

Notebook di test manuale/esplorativo per il domain layer.

Copre `Rarity`, `Card`, `SourcePackRef`, `PackDefinition`, `PlayerDefinition`,
`OpeningPolicy` e `ProjectDefinition`.


## 1. Import

In [1]:
from ygomaster_packbuilder.domain import (
    Rarity, Card,
    SourceProvider, SourcePackRef,
    PackType, PackDefinition,
    PlayerDefinition,
    RarityDistribution, TenPackGuarantee, UltraRarePity, OpeningPolicy,
    ProjectDefinition,
)

print("Domain imports OK")

Domain imports OK


## 2. Rarity

In [2]:
assert Rarity.N.value == "N"
assert Rarity.R.value == "R"
assert Rarity.SR.value == "SR"
assert Rarity.UR.value == "UR"

print(list(Rarity))

[<Rarity.N: 'N'>, <Rarity.R: 'R'>, <Rarity.SR: 'SR'>, <Rarity.UR: 'UR'>]


## 3. Card: stato iniziale e arricchimento

In [3]:
card = Card(
    name='"A" Cell Breeding Device',
    ydk_id=34541863,
    source_rarity="Common",
)

assert card.master_duel_id is None
assert card.rarity is None

card.master_duel_id = 12345
card.rarity = Rarity.N

assert card.master_duel_id == 12345
assert card.rarity is Rarity.N

card

Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=12345, rarity=<Rarity.N: 'N'>)

## 4. SourcePackRef: immutabilità

In [4]:
source = SourcePackRef(
    provider=SourceProvider.YGOPRODECK,
    external_id="24116",
)

assert source.external_id == "24116"

try:
    source.external_id = "99999"
    raise AssertionError("SourcePackRef dovrebbe essere frozen")
except Exception as exc:
    print(type(exc).__name__, "-", exc)

FrozenInstanceError - cannot assign to field 'external_id'


## 5. PackDefinition: main pack

In [5]:
main_pack = PackDefinition(
    key="livio-scaled-invasion",
    name="Scaled Invasion",
    description="Personal main pack",
    type=PackType.PERSONAL_MAIN,
    source=source,
    expected_card_count=100,
    booster_size=7,
    opening_policy_key="configuration-a",
    owner_key="livio",
)

assert main_pack.parent_pack_key is None
assert main_pack.cards == []

main_pack.cards.append(card)
assert main_pack.cards[0].ydk_id == 34541863

main_pack

PackDefinition(key='livio-scaled-invasion', name='Scaled Invasion', description='Personal main pack', type=<PackType.PERSONAL_MAIN: 'personal_main'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=100, booster_size=7, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key=None, cards=[Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=12345, rarity=<Rarity.N: 'N'>)])

## 6. Tre mini-pack fratelli

In [6]:
mini_sources = [
    SourcePackRef(SourceProvider.YGOPRODECK, "24117"),
    SourcePackRef(SourceProvider.YGOPRODECK, "24118"),
    SourcePackRef(SourceProvider.YGOPRODECK, "24119"),
]

mini_keys = ["livio-alien", "livio-venom", "livio-ogdoadic"]

mini_packs = [
    PackDefinition(
        key=key,
        name=key.replace("-", " ").title(),
        description="Targeted personal mini-pack",
        type=PackType.PERSONAL_MINI,
        source=src,
        expected_card_count=20,
        booster_size=3,
        opening_policy_key="configuration-a",
        owner_key="livio",
        parent_pack_key="livio-scaled-invasion",
    )
    for key, src in zip(mini_keys, mini_sources)
]

assert len(mini_packs) == 3
assert all(p.type is PackType.PERSONAL_MINI for p in mini_packs)
assert all(p.parent_pack_key == main_pack.key for p in mini_packs)

mini_packs

[PackDefinition(key='livio-alien', name='Livio Alien', description='Targeted personal mini-pack', type=<PackType.PERSONAL_MINI: 'personal_mini'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24117'), expected_card_count=20, booster_size=3, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key='livio-scaled-invasion', cards=[]),
 PackDefinition(key='livio-venom', name='Livio Venom', description='Targeted personal mini-pack', type=<PackType.PERSONAL_MINI: 'personal_mini'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24118'), expected_card_count=20, booster_size=3, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key='livio-scaled-invasion', cards=[]),
 PackDefinition(key='livio-ogdoadic', name='Livio Ogdoadic', description='Targeted personal mini-pack', type=<PackType.PERSONAL_MINI: 'personal_mini'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodec

## 7. Shared staples pack

In [7]:
staples_pack = PackDefinition(
    key="shared-staples",
    name="Shared Staples",
    description="Shared staple card pool",
    type=PackType.SHARED_STAPLES,
    source=SourcePackRef(SourceProvider.YGOPRODECK, "25000"),
    expected_card_count=200,
    booster_size=11,
    opening_policy_key="configuration-a",
)

assert staples_pack.owner_key is None
assert staples_pack.parent_pack_key is None

staples_pack

PackDefinition(key='shared-staples', name='Shared Staples', description='Shared staple card pool', type=<PackType.SHARED_STAPLES: 'shared_staples'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='25000'), expected_card_count=200, booster_size=11, opening_policy_key='configuration-a', owner_key=None, parent_pack_key=None, cards=[])

## 8. PlayerDefinition

In [8]:
player = PlayerDefinition(
    key="livio",
    display_name="Livio",
    main_pack_key=main_pack.key,
    mini_pack_keys=[p.key for p in mini_packs],
)

assert len(player.mini_pack_keys) == 3
player

PlayerDefinition(key='livio', display_name='Livio', main_pack_key='livio-scaled-invasion', mini_pack_keys=['livio-alien', 'livio-venom', 'livio-ogdoadic'])

## 9. Configurazione A

In [9]:
configuration_a = OpeningPolicy(
    key="configuration-a",
    normal_slot_distribution=RarityDistribution(
        normal=55.0,
        rare=35.0,
        super_rare=7.5,
        ultra_rare=2.5,
    ),
    ten_pack_guarantee=TenPackGuarantee(
        enabled=True,
        final_slot_distribution=RarityDistribution(
            normal=0.0,
            rare=0.0,
            super_rare=80.0,
            ultra_rare=20.0,
        ),
    ),
    ultra_rare_pity=UltraRarePity(
        enabled=True,
        trigger_after_ten_pack_without_ur=True,
        guaranteed_slot_distribution=RarityDistribution(
            normal=0.0,
            rare=0.0,
            super_rare=0.0,
            ultra_rare=100.0,
        ),
    ),
)

assert configuration_a.normal_slot_distribution.ultra_rare == 2.5
assert configuration_a.ten_pack_guarantee.final_slot_distribution.super_rare == 80.0
assert configuration_a.ultra_rare_pity.guaranteed_slot_distribution.ultra_rare == 100.0

try:
    configuration_a.key = "changed"
    raise AssertionError("OpeningPolicy dovrebbe essere frozen")
except Exception as exc:
    print(type(exc).__name__, "-", exc)

configuration_a

FrozenInstanceError - cannot assign to field 'key'


OpeningPolicy(key='configuration-a', normal_slot_distribution=RarityDistribution(normal=55.0, rare=35.0, super_rare=7.5, ultra_rare=2.5), ten_pack_guarantee=TenPackGuarantee(enabled=True, final_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=80.0, ultra_rare=20.0)), ultra_rare_pity=UltraRarePity(enabled=True, trigger_after_ten_pack_without_ur=True, guaranteed_slot_distribution=RarityDistribution(normal=0.0, rare=0.0, super_rare=0.0, ultra_rare=100.0)))

## 10. ProjectDefinition completo

In [10]:
project = ProjectDefinition(
    name="Master Duel Custom League",
    players=[player],
    packs=[main_pack, *mini_packs, staples_pack],
    opening_policies=[configuration_a],
)

assert len(project.players) == 1
assert len(project.packs) == 5
assert len(project.opening_policies) == 1

project

ProjectDefinition(name='Master Duel Custom League', players=[PlayerDefinition(key='livio', display_name='Livio', main_pack_key='livio-scaled-invasion', mini_pack_keys=['livio-alien', 'livio-venom', 'livio-ogdoadic'])], packs=[PackDefinition(key='livio-scaled-invasion', name='Scaled Invasion', description='Personal main pack', type=<PackType.PERSONAL_MAIN: 'personal_main'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24116'), expected_card_count=100, booster_size=7, opening_policy_key='configuration-a', owner_key='livio', parent_pack_key=None, cards=[Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=12345, rarity=<Rarity.N: 'N'>)]), PackDefinition(key='livio-alien', name='Livio Alien', description='Targeted personal mini-pack', type=<PackType.PERSONAL_MINI: 'personal_mini'>, source=SourcePackRef(provider=<SourceProvider.YGOPRODECK: 'ygoprodeck'>, external_id='24117'), expected_card_count=20, booster_si

## 11. Controlli strutturali

In [11]:
packs_by_key = {p.key: p for p in project.packs}
players_by_key = {p.key: p for p in project.players}
policies_by_key = {p.key: p for p in project.opening_policies}

assert len(packs_by_key) == len(project.packs)
assert len(players_by_key) == len(project.players)
assert len(policies_by_key) == len(project.opening_policies)

assert player.main_pack_key in packs_by_key
assert all(k in packs_by_key for k in player.mini_pack_keys)
assert all(p.opening_policy_key in policies_by_key for p in project.packs)

for mini in mini_packs:
    assert mini.parent_pack_key == player.main_pack_key
    assert mini.owner_key == player.key

print("Structural checks OK")

Structural checks OK


## 12. Regola futura: mini-pack fratelli disgiunti

Per due mini-pack fratelli deve valere:

$M_i \cap M_j = \varnothing$

Il confronto viene fatto tramite `Card.ydk_id`.


In [12]:
def card_ids(pack: PackDefinition) -> set[int]:
    return {card.ydk_id for card in pack.cards}

for i, left in enumerate(mini_packs):
    for right in mini_packs[i + 1:]:
        overlap = card_ids(left) & card_ids(right)
        assert not overlap, (
            f"Duplicate cards between sibling mini-packs "
            f"{left.key} and {right.key}: {overlap}"
        )

print("Sibling mini-pack overlap check OK")

Sibling mini-pack overlap check OK
